In [ ]:
%load_ext autoreload
%autoreload 2
    
import polars as pl
from polars import col
from polars import selectors as cs
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.figure_factory as ff
import time
import numpy as np
from jsr.utils.processing import process_dataframes, _find_shift_ids
from jsr.utils.col_namespace import Sched, Ship, Trans
from plotly.subplots import make_subplots
from jsr.plotting.plot_shipments import *
from jsr.utils.plotting import plot_write_image
from datetime import timedelta

In [ ]:
dataframes = process_dataframes("../data")
df_schedule = dataframes["schedule"]
df_shipments = dataframes["shipments"]
df_transactions = dataframes["transactions"]

## Predicting Total # of Transactions per day

The goal is to try to predict the total number of transactions each day. Using summarized data

### Processing

In [ ]:
from sktime.transformations.series.summarize import WindowSummarizer
from sktime.transformations.compose import YtoX
from sktime.forecasting.compose import make_reduction
from sklearn.ensemble import GradientBoostingRegressor
from sktime.transformations.series.fourier import FourierFeatures
from sktime.transformations.compose import TransformerPipeline, FeatureUnion
from sktime.transformations.series.impute import Imputer
from sktime.forecasting.model_selection import temporal_train_test_split
from sktime.forecasting.base import ForecastingHorizon

In [ ]:
# Grapping operator counts using sliding window
df_transactions_total = ( df_transactions
  .sort(Trans.START_TIME)
  .group_by_dynamic(Trans.START_TIME, every="1d")
  .agg(col(Trans.OPERATOR_ID).unique().count())
)

# Converting to period indexed pandas series
y = df_transactions_total[Trans.OPERATOR_ID].to_numpy()
y_dates = pd.to_datetime(df_transactions_total[Trans.START_TIME].to_numpy())
y = pd.Series(y, index=y_dates.to_period('d'))
y.index.name = "time"

# Fill in missing days (as 0's)
full_index = pd.period_range(start=y.index.min(), end=y.index.max(), freq='D')
y = y.reindex(full_index,fill_value=0)

# Promote to dataframe and add categorical feature: days of week
df = pd.DataFrame(y, columns=["value"])
df['day_of_week'] = df.index.dayofweek.astype('category')
df = pd.get_dummies(df, columns=['day_of_week'], prefix='dow', drop_first=True)
df = df.asfreq('D').astype('float32')

# Train/test split (we don't have a lot of data to work with...)
train, test = temporal_train_test_split(df, test_size=0.3)
X_train, X_test = train.drop(columns=['value']), test.drop(columns=['value'])
y_train, y_test = train['value'], test['value']

### Feature Extraction

In [ ]:
# Features based off of moving window over original values
window_summarizer = WindowSummarizer(

    # Create time-shifted and smoothed versions of original values
    lag_feature={
        "lag": [1,2,3,4,5],
        "mean": [[1, 3], [1, 5], [1, 7]],
        "std": [[1,4]],
    }
)


# Branch 1: Fourier derived from days of week
day_branch = TransformerPipeline([
    ("fourier_exog", FourierFeatures(sp_list=[12, 24*7], fourier_terms_list=[10, 5])),
])

# Branch 2: Summary features from y (values)
summary_branch = TransformerPipeline([
    ("summary_exog", window_summarizer),
    # need to impute to due NaN values introduces from summarizer
    ("imputer", Imputer(method="constant", value=0.0))
])

# Branch 3: Fourier derived from y (values)
y_as_exog_branch = TransformerPipeline([
    ("ytox", YtoX()),                            
    ("fourier_y", FourierFeatures(sp_list=[12, 24*7], fourier_terms_list=[10, 5])),
])

# Merge branches
combined_exog = FeatureUnion([
    ("fourier_day_feature", day_branch),
    ("fourier_y_feature", y_as_exog_branch),
    ("summary_feature", summary_branch),
])

# Show pipeline
combined_exog

In [ ]:
# Extract the features from the pipeline so we can use them downstream
combined_exog.fit(X=X_train, y=y_train)
tX_train = combined_exog.transform(X=X_train, y=y_train)
tX_test = combined_exog.transform(X=X_test, y=y_test)

In [ ]:
# Extract just the day features to evaluate them separately
day_branch.fit(X=X_train, y=y_train)
tXd_train = day_branch.transform(X=X_train, y=y_train)
tXd_test = day_branch.transform(X=X_test, y=y_test)

## Forecasting

### Model

In [ ]:
forecaster = make_reduction(
    GradientBoostingRegressor(random_state=123),
    strategy="recursive",
    pooling="local",
    window_length=10,
)

### Test 1: Fit model using only shipment values

In [ ]:
forecaster.fit(y=y_train, fh=y_test.index)
preds_baseline = forecaster.predict(fh=y_test.index)
farout_preds_baseline = forecaster.predict(fh=range(20, 60))
print("MSE:", ((preds_baseline - y_test)**2).mean()) 

### Test 2: Fit model also using day of week data

In [ ]:
forecaster.fit(y=y_train, X=X_train, fh=y_test.index)
preds_with_dow = forecaster.predict(fh=y_test.index, X=X_test)
farout_preds_with_dow = forecaster.predict(fh=range(20, 60), X=X_test)
print("MSE:", ((preds_with_dow - y_test)**2).mean()) 

### Test 3: Fit model using all transformed features

In [ ]:
forecaster.fit(y=y_train, X=tX_train, fh=y_test.index)
preds_with_transformed = forecaster.predict(fh=y_test.index, X=tX_test)
farout_preds_with_transformed = forecaster.predict(fh=range(20, 60), X=tX_test)
print("MSE:", ((preds_with_transformed - y_test)**2).mean())

## Visualization of forecast

In [ ]:
original = pd.concat([y_train, y_test, 
                     farout_preds_baseline * 0  ]) # fake extended values

predicted_with_baseline = pd.concat([y_train, preds_baseline, farout_preds_baseline])
predicted_with_dow = pd.concat([y_train, preds_with_dow, farout_preds_with_dow])
predicted_with_transformed = pd.concat([y_train, preds_with_transformed, farout_preds_with_transformed])

# Prepare DF for plotting with px
forecast_df = pd.DataFrame({
    "Date": original.index.to_timestamp(),
    "Ground Truth": original,
    "No Features": predicted_with_baseline, 
    "+ DoW": predicted_with_dow,
    "+ Additional": predicted_with_transformed})
forecast_df = forecast_df.set_index('Date')

forecast_df = (
    forecast_df
        .reset_index()
        .melt(id_vars='Date', var_name='Model', value_name='Total Operators')
)

# only interested in dates that begin where predictions diverge from original
cutoff = pd.to_datetime('2020-11-16')
last_day = y_test.index[-1].to_timestamp()
last_train_data = y_train.index[-1].to_timestamp()
forecast_near_end = forecast_df[forecast_df["Date"] > cutoff]
forecast_near_end = forecast_near_end[forecast_near_end["Date"] < last_day + timedelta(days=3)]


forecast_beyond = forecast_df[forecast_df["Date"] > last_day - timedelta(days=3) ]

In [ ]:
fig = px.line(forecast_near_end, x='Date',
    y='Total Operators',
    color='Model',
    title="Comparing Forecasts of Total Daily Operators")


fig.update_traces(
    selector=dict(name='Ground Truth'),
    line=dict(dash='dot'),
    opacity=0.3
)

fig.update_layout(
        margin=dict(l=20, r=20, t=40, b=30),
        xaxis_title_font_size=18,
        yaxis_title_font_size=18,
        title_y=0.99,
        title_x=0.5,
        title_font_size=32,
        font_family="Overpass")

# change legend to free up space...
fig.update_layout(
    legend=dict(
        yanchor="top",
        y=1.0,     
        xanchor="right",
        x=0.85,
        orientation="h",
        bordercolor="Black",          
        borderwidth=2,
    )
)


# Cutoff: last day of training data that models have access to
fig.add_vline(x=last_train_data, line_color="black", line_dash="dot")


# Cutoff: last day we have actual data for
fig.add_vline(x=last_day, line_color="black", line_dash="dot")

# Create version where we hide forecasts
names_to_hide = ['No Features', '+ DoW', '+ Additional']
for name in names_to_hide:
    fig.update_traces(visible='legendonly', selector=dict(name=name))   
plot_write_image(fig, "transaction_comparing_forecasts_hidden", scale=4)

# Create one where we include baseline
fig.update_traces(visible=True, selector=dict(name="No Features"))
plot_write_image(fig, "transaction_comparing_forecasts_baseline", scale=4)
                  
# Create final one
for name in names_to_hide:
    fig.update_traces(visible=True, selector=dict(name=name))   
plot_write_image(fig, "transaction_comparing_forecasts_all", scale=4)
fig

In [ ]:
fig = px.line(forecast_beyond, x='Date',
    y='Total Operators',
    color='Model',
    title="Extended Forecast (past Dec 8)")


fig.update_traces(
    selector=dict(name='Ground Truth'),
    line=dict(dash='dot'),
    opacity=0.3
)

fig.update_layout(
        margin=dict(l=20, r=20, t=40, b=30),
        xaxis_title_font_size=18,
        yaxis_title_font_size=18,
        title_y=0.99,
        title_x=0.5,
        title_font_size=32,
        font_family="Overpass")

# change legend to free up space...
fig.update_layout(
    legend=dict(
        yanchor="top",
        y=1.0,     
        xanchor="right",
        x=0.85,
        orientation="h",
        bordercolor="Black",          
        borderwidth=2,
    )
)

# Cutoff: last day we have actual data for
fig.add_vline(x=last_day, line_color="black", line_dash="dot")

plot_write_image(fig, "transaction_extended_forecast", scale=4)
fig